# Compare retrieval techniques

**Goal:** Read the actual retrieval algorithms and reproduce the 31-configuration comparison from saved results.

Run cells from top to bottom. Default cells work offline; model execution is an explicit opt-in and writes only to ignored `outputs/`.

## 1. Set up paths

Find the repository and load the analysis helpers.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "data/final").is_dir(), "Run from the repository or notebooks folder"
sys.path.insert(0, str(ROOT / "src"))
import pandas as pd
import matplotlib.pyplot as plt
from sleepinn_study.io import read_json, read_jsonl, output_directory
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})


## 2. Inspect the algorithms

Dense retrieval uses BGE-M3; BM25 uses lexical matches. Balanced merge alternates ranks, while RRF sums reciprocal ranks (constant 60). SCOPE adds structural headers; MiniLM reranks only configurations marked rerank. Jiwar adds neighbouring passages. Multi-query variants use three Gemma-generated queries. The selected SCOPE–Jiwar pipeline omits multi-query expansion.

In [ ]:
import inspect
from sleepinn_rag import september_core as core
from sleepinn_rag.kb.scope import build_contextualized_chunk
display(pd.DataFrame(core.strategy_matrix(ROOT / "src")))
print(inspect.getsource(core.fuse_lists))
print(inspect.getsource(core.make_blocks))
print(inspect.getsource(core.apply_budget))
print("Full dense/BM25/reranking loop: src/sleepinn_study/retrieval_experiment.py")

## 3. Run a new retrieval comparison when sources are available

The runner prepares chunks and indexes, creates alternative queries, executes the configurations and reports evidence recall. It requires licensed PDFs, the evidence-enriched bank and the pinned GPU environment. Missing source evidence causes a clear error. This cell is disabled for an offline read-through.

In [ ]:
RUN_RETRIEVAL = False
if RUN_RETRIEVAL:
    import subprocess
    subprocess.run([sys.executable, "-m", "sleepinn_study.retrieval_experiment", "--run-id", "retrieval_new_001", "--stage", "all"], cwd=ROOT, check=True)
else:
    print("See REPRODUCING.md for local input paths and installation.")

## 4. Plot all 31 configurations

Evidence recall is distinct from answer correctness. Configurations were compared on the full knowledge bank; these are exploratory comparisons, not a held-out retrieval test.

In [ ]:
r = pd.read_csv(ROOT / "results/retrieval/retrieval_benchmark.csv")
assert len(r) == 31 * 1215
means = r.groupby("strategy")["budgeted_evidence_recall@5"].mean().sort_values()
fig, ax = plt.subplots(figsize=(10, 11))
colors = ["#d88735" if s == "scope_hybrid_rrf_rerank_jiwar" else "#327c9d" for s in means.index]
ax.barh(means.index, means * 100, color=colors)
ax.set(xlabel="Evidence recall within five budgeted blocks (%)", xlim=(0, 100))
fig.tight_layout()
fig.savefig(output_directory("figures") / "retrieval_all_configurations.png")
plt.show()